# Lab 07 — 05 Great Expectations



In [0]:
%pip install -q "great-expectations>=1.5,<2"


In [0]:
from pathlib import Path
import sys
cwd=Path.cwd().resolve()
project_root=next((p for p in [cwd,*cwd.parents] if (p/'src'/'lab07').exists()),None)
if project_root and str(project_root/'src') not in sys.path: sys.path.insert(0,str(project_root/'src'))
if project_root and str(project_root/'tools') not in sys.path: sys.path.insert(0,str(project_root/'tools'))
dbutils.widgets.text('catalog','dbr_dev','01 Catalog'); dbutils.widgets.text('schema','parvinbadalov','02 Schema'); dbutils.widgets.text('volume_name','lab07_data_quality','03 Volume'); dbutils.widgets.text('run_id','manual','04 Run ID')
catalog=dbutils.widgets.get('catalog'); schema=dbutils.widgets.get('schema'); volume_name=dbutils.widgets.get('volume_name'); run_id=dbutils.widgets.get('run_id')
assert catalog=='dbr_dev' and schema=='parvinbadalov', f'Lab 07 requires dbr_dev.parvinbadalov, got {catalog}.{schema}'
volume_root=f'/Volumes/{catalog}/{schema}/{volume_name}'


import importlib
import inspect
import lab07.gx_runner as gx_runner

importlib.invalidate_caches()
importlib.reload(gx_runner)

print(gx_runner.__file__)
print(inspect.getsource(gx_runner.run_suite))

load_suite_spec = gx_runner.load_suite_spec
run_suite = gx_runner.run_suite

In [0]:
from lab07.gx_runner import load_suite_spec,run_suite
for layer,table,file in [('bronze','business_license_bronze','bronze_license_suite.json'),('silver','business_license_validated','silver_license_suite.json'),('gold','license_quality_daily','gold_license_suite.json')]:
    result=run_suite(spark.table(f'{catalog}.{schema}.{table}'),load_suite_spec(project_root/'dq'/'great_expectations'/file)); print(layer,result.success); assert bool(result.success)
print('GX MEDALLION VALIDATION: PASS')
